Импорт библиотек

In [50]:
from random import randint

Глобальные переменные

In [51]:
# Эти переменные объявлены глобально, потому что они задействуются почти во всех функциях CМО (система массового обслуживания)
processing_queue_len = 0            # Размер очереди поступивших и необработанных заявок

state = 'free'                      # Статус обработчика

Генератор очереди заявок

In [52]:
# Симулирует случайное поступление заявок в реальном времени.
# Элемент очереди - это время поступление новой заявки
# Возвращает список с заявками

def QueueGenerator(a, b, max_time):
    processing_queue_len = randint(1000, 2000)
    time = 0
    queue = []
    for _ in range(processing_queue_len):
        new_app = time + randint(a, b)
        if new_app < max_time:
            queue.append(new_app)
            time = new_app
    return queue

Получение заявки

In [53]:
# Принимает заявку
# Высчитывает временную метку когда заявка будет обработана с учетом уже имеющихся в очереди и
# возвращает эту временную метку

def ApplicationReceiption(current_time, last_app_time, c, d):    
    global state, processing_queue_len
    event = max(current_time, last_app_time) + randint(c, d)    # Определяем временную метку, когда заявка будет обработана
    if state == 'free':
        state = 'busy'
    else:
        processing_queue_len += 1
    return event

Обработка заявки

In [54]:
# Принимает заявку из очереди необработанных заявок
# Регулирует состояние обработчика

def ApplicationProcessing(max_time, current_time):    
    global processing_queue_len, state
    if current_time > max_time:
        return
    if processing_queue_len > 0:
        processing_queue_len -= 1
        state = 'busy'
    else:
        state = 'free'

Основная функция

In [ ]:
# Запускает работу СМО
# receiption_queue - очередь всех заявок
# processing_queue - очередь необработанных заявок

def MassServiceSystem(receiption_queue, processing_queue, a, b, c, d):
    last_app_time = 0           # Время завершения обработки последней заявки в очереди поступивших заявок processing_queue
    max_time = 500              # Максимальое время работы системы
    current_time = 0            # Текущий момет времени в системе

    processed_apps = 0          # Обработанные заявки
    unfinished_apps = 0         # Незавершенные заявки

    while current_time < max_time:        # Система работает до момента времени current_time

        receiption_time = -1       # Эти 2 переменные необходимы для выбора, из какой очереди брать событие -
                                   # из очереди поступающих (receiption_queue), или из очереди необработанных (processing_queue)
        processing_time = -1

        # В цепочке try-except присваиваем значения для receiption_time и processing_time
        try:                            
            receiption_time = receiption_queue[0]
        except IndexError:
            pass 
        try:
            processing_time = processing_queue[0]
        except IndexError:
            pass

        # Затем, сравнивая receiption_time и processing_time, выбираем какое событие обрабатывать
        if len(receiption_queue) and len(processing_queue) and receiption_time < processing_time:
            new_app_time = receiption_queue.pop(0)
            code = 1
        else:
            try:
                new_app_time = processing_queue.pop(0)
                code = 2
            except IndexError:
                new_app_time = receiption_queue.pop(0)
                code = 1
        
        # Обновляем время системы current_time
        current_time = new_app_time
        if code == 1:           # Код 1 - получаем новую заявку из receiption_queue
            event = ApplicationReceiption(current_time, last_app_time, c, d)
            unfinished_apps += 1
            last_app_time = event
            processing_queue.append(event)
        elif code == 2:         # Код 2 - обрабатываем заявку из processing_queue
            ApplicationProcessing(max_time, current_time)
            processed_apps += 1
            unfinished_apps -= 1
    else:
        print('Время работы системы истекло. Завершение работы')
        print()
        return processed_apps, unfinished_apps, processing_queue  # По завершении работы возвращаем кол-во всех обработанных и необработанных заявок

In [56]:
processing_queue_len = 0       # Здесь обновляются значения глобальных переменных при новом вызове функции
state = 'free'

max_time = 500

a = 15     # Минимальный промежуток времени между поступлением заявок
b = 30     # Максимальный промежуток времени между поступлением заявок
c = 20     # Минимальное время обработки заявок
d = 30     # Максимальное время обработки заявок

receiption_queue = QueueGenerator(a, b, max_time)   # Генерация receiption_queue
processing_queue = []                                   # processing_queue изначально пустой

print('Очередь поступающих заявок')
print(receiption_queue)

result = MassServiceSystem(receiption_queue, processing_queue, a, b, c, d) # Запускаем СМО

print()
print(f'Всего поступивших заявок: {result[0] + result[1]}')
print(f'Обработанные: {result[0]}')
print(f'Необработанные: {result[1]}')
print(f'Временные метки необработанных заявок: {result[2]}')

Очередь поступающих заявок
[18, 40, 64, 83, 109, 131, 147, 171, 201, 224, 242, 265, 287, 309, 329, 357, 387, 404, 430, 446, 473, 490]
Время работы системы истекло. Завершение работы


Всего поступивших заявок: 22
Обработанные: 19
Необработанные: 3
Временные метки необработанных заявок: [525, 550, 579]
